In [5]:
import torch
import torch.fx
import os
from detectron2.data.datasets import register_coco_instances
from detectron2.data import MetadataCatalog, DatasetCatalog
from dotenv import load_dotenv
load_dotenv() # Ensure Mac GPU fallback is active

from detectron2.engine import DefaultTrainer
from detectron2.config import get_cfg
from detectron2 import model_zoo
import os
from detectron2.engine import DefaultPredictor
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

In [6]:
base_ml_dir = os.path.abspath("../data/ml_dataset_cropped")

# register the training and validation sets
register_coco_instances(
    "chaos_train", 
    {}, 
    os.path.join(base_ml_dir, "train", "train_coco.json"), 
    os.path.join(base_ml_dir, "train", "images")
)

register_coco_instances(
    "chaos_val", 
    {}, 
    os.path.join(base_ml_dir, "val", "val_coco.json"), 
    os.path.join(base_ml_dir, "val", "images")
)

# verify that they were successfully registered
chaos_metadata = MetadataCatalog.get("chaos_train")
dataset_dicts = DatasetCatalog.get("chaos_train")
print(f"Successfully registered {len(dataset_dicts)} training images.")

[10/09 09:25:20 d2.data.datasets.coco]: Loaded 1170 images in COCO format from /Users/jjwill18/Desktop/europa-chaos-blocks/data/ml_dataset_cropped/train/train_coco.json
Successfully registered 1170 training images.


In [7]:
# define folder for storing results
base_run_dir = os.path.abspath("../results/runs")
# 1. Load the Default Configuration
cfg = get_cfg()

# load the architecture for Mask R-CNN with a ResNet-50 backbone
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))

# defune training dataset
cfg.DATASETS.TRAIN = ("chaos_train",)
cfg.DATASETS.TEST = ()  

# set hardware settings
cfg.MODEL.DEVICE = "mps"          # use the Apple Silicon GPU
cfg.DATALOADER.NUM_WORKERS = 2    # how many CPU cores to use for loading images
cfg.SOLVER.IMS_PER_BATCH = 2      # small batch size

# set learning rate and max iterations
cfg.SOLVER.BASE_LR = 0.001
cfg.SOLVER.MAX_ITER = 10000       
cfg.SOLVER.STEPS = (6000, 8000) # Decay learning rate at these iterations to help it "fine-tune"

# model specifics
# Start with the pre-trained earth weights
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128   # Number of regions to propose per image
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1              # We only have 1 class: "chaos_block"

# set the output directory
cfg.OUTPUT_DIR = os.path.join(base_run_dir, "run_10_maskrcnn")
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

print("Starting Mask R-CNN Training...")
trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()
print("Finished.")

Starting Mask R-CNN Training...
[10/09 09:26:20 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-

Skip loading parameter 'roi_heads.box_predictor.cls_score.weight' to the model due to incompatible shapes: (81, 1024) in the checkpoint but (2, 1024) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.cls_score.bias' to the model due to incompatible shapes: (81,) in the checkpoint but (2,) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.bbox_pred.weight' to the model due to incompatible shapes: (320, 1024) in the checkpoint but (4, 1024) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.bbox_pred.bias' to the model due to incompatible shapes: (320,) in the checkpoint but (4,) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.mask_head.predictor.weight' to the model due to incompatible shapes: (80, 256, 1, 1) in the checkpoint but (1, 256, 1, 1) in

[10/09 09:26:21 d2.engine.train_loop]: Starting training from iteration 0
[10/09 09:26:35 d2.utils.events]:  eta: 1:34:51  iter: 19  total_loss: 3.151  loss_cls: 0.8394  loss_box_reg: 0.42  loss_mask: 0.6934  loss_rpn_cls: 1.039  loss_rpn_loc: 0.08374    time: 0.5654  last_time: 0.5549  data_time: 0.1185  last_data_time: 0.0009   lr: 1.9981e-05  
[10/09 09:26:47 d2.utils.events]:  eta: 1:37:29  iter: 39  total_loss: 2.274  loss_cls: 0.6136  loss_box_reg: 0.5548  loss_mask: 0.6876  loss_rpn_cls: 0.2824  loss_rpn_loc: 0.0685    time: 0.5931  last_time: 0.6862  data_time: 0.0008  last_data_time: 0.0008   lr: 3.9961e-05  
[10/09 09:26:59 d2.utils.events]:  eta: 1:38:54  iter: 59  total_loss: 2.116  loss_cls: 0.5041  loss_box_reg: 0.5879  loss_mask: 0.6784  loss_rpn_cls: 0.2528  loss_rpn_loc: 0.06057    time: 0.5982  last_time: 0.6263  data_time: 0.0007  last_data_time: 0.0007   lr: 5.9941e-05  
[10/09 09:27:12 d2.utils.events]:  eta: 1:39:28  iter: 79  total_loss: 2.149  loss_cls: 0.5033  

In [8]:
# evaluate model metrics
print("Loading the trained model for evaluation...")

# update the configuration to point to your finished weights
cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.50   # Only accept predictions with > 50% confidence
cfg.DATASETS.TEST = ("chaos_val", )

# build the Predictor
predictor = DefaultPredictor(cfg)

# setup the COCO Evaluator
# This compares the predictions against your 'val_coco.json'
evaluator = COCOEvaluator(
    "chaos_val", 
    output_dir=cfg.OUTPUT_DIR, 
    use_fast_impl=False # Slower, but more accurate math
)
val_loader = build_detection_test_loader(cfg, "chaos_val")

# run the inference and evaluation
print("\nRunning inference on the Validation Set...")
eval_results = inference_on_dataset(predictor.model, val_loader, evaluator)

print("\n==============================================")
print("FINAL EVALUATION RESULTS")
print("==============================================")

# extract and print the two most important metrics
bbox_map50 = eval_results['bbox']['AP50']
segm_map50 = eval_results['segm']['AP50']

print(f"Bounding Box mAP50: {bbox_map50:.2f} %")
print(f"Polygon Mask mAP50: {segm_map50:.2f} %")
print("==============================================")

Loading the trained model for evaluation...
[10/09 11:03:48 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /Users/jjwill18/Desktop/europa-chaos-blocks/results/runs/run_10_maskrcnn/model_final.pth ...
[10/09 11:03:49 d2.data.datasets.coco]: Loaded 36 images in COCO format from /Users/jjwill18/Desktop/europa-chaos-blocks/data/ml_dataset_cropped/val/val_coco.json
[10/09 11:03:49 d2.data.dataset_mapper]: [DatasetMapper] Augmentations used in inference: [ResizeShortestEdge(short_edge_length=(800, 800), max_size=1333, sample_style='choice')]
[10/09 11:03:49 d2.data.common]: Serializing the dataset using: <class 'detectron2.data.common._TorchSerializedList'>
[10/09 11:03:49 d2.data.common]: Serializing 36 elements to byte tensors and concatenating them all ...
[10/09 11:03:49 d2.data.common]: Serialized dataset takes 0.07 MiB

Running inference on the Validation Set...
[10/09 11:03:49 d2.evaluation.evaluator]: Start inference on 36 batches
[10/09 11:03:53 d2.evaluat